# Configuration Management — dask_setup Recipe

This notebook demonstrates advanced configuration management in `dask_setup`, covering:

- Using built-in profiles (`climate_analysis`, `interactive`, `development`, etc.)
- Building explicit `DaskSetupConfig` objects with full parameter control
- Loading custom YAML profiles
- Using `ConfigManager` to list, inspect and manage profiles
- Configuration best practices

> **Note:** Each demo creates and closes its own cluster so resources are fully independent between examples.

## Setup and Imports

In [ ]:
import sys
import time
from pathlib import Path

import dask
import dask.array as da
import numpy as np

from dask_setup import setup_dask_client
from dask_setup.config import DaskSetupConfig
from dask_setup.config_manager import ConfigManager

print("Imports OK")

## 1. Built-in Profiles

`dask_setup` ships a set of pre-tuned profiles for common scientific computing patterns.
Use `ConfigManager` to list and inspect them.

In [ ]:
manager = ConfigManager()
profiles = manager.list_profiles()
builtin = {name: p for name, p in profiles.items() if p.builtin}

print(f"Built-in profiles ({len(builtin)}):")
for name, p in builtin.items():
    cfg = p.config
    print(f"  {name:20s}  workload={cfg.workload_type:6s}  reserve={cfg.reserve_mem_gb} GB  "
          f"adaptive={cfg.adaptive}")

In [ ]:
# Use a built-in profile — development is safe to run anywhere
client, cluster, tmp = setup_dask_client(profile="development")

workers = client.scheduler_info()["workers"]
print(f"Workers: {len(workers)}")
print(f"Temp dir: {tmp}")

# Quick benchmark
t0 = time.perf_counter()
x = da.random.random((2000, 2000), chunks=(500, 500))
result = (x + x.T).mean().compute()
elapsed = time.perf_counter() - t0

print(f"Result: {result:.6f}  wall={elapsed:.2f}s")

client.close(); cluster.close()

## 2. Explicit `DaskSetupConfig` Object

For reproducible, version-controlled configurations, build a `DaskSetupConfig` directly.
Every parameter has a sensible default, so you only need to specify what differs from the base.

In [ ]:
config = DaskSetupConfig(
    workload_type="mixed",
    max_workers=4,
    reserve_mem_gb=30.0,
    spill_compression="lz4",
    memory_target=0.70,
    memory_spill=0.80,
    dashboard=True,
    description="Custom mixed workload — 4 workers, lz4 spill",
)

print("Config object:")
print(f"  workload_type   : {config.workload_type}")
print(f"  max_workers     : {config.max_workers}")
print(f"  reserve_mem_gb  : {config.reserve_mem_gb}")
print(f"  spill_compression: {config.spill_compression}")
print(f"  memory_target   : {config.memory_target}")

client, cluster, tmp = setup_dask_client(config=config)
workers = client.scheduler_info()["workers"]
print(f"\nCluster started — {len(workers)} workers, tmp={tmp}")

# Demonstration workload
data = da.random.random((1500, 1500), chunks=(300, 300))
result = (data * 2 + da.sin(data)).mean().compute()
print(f"Result: {result:.6f}")

client.close(); cluster.close()

## 3. Inline `dask.config` Overrides

For fine-grained Dask scheduler/worker settings, use `dask.config.set()` as a context manager
alongside `setup_dask_client`.

In [ ]:
with dask.config.set({
    "distributed.scheduler.idle-timeout": "30s",
    "distributed.worker.daemon": False,
    "distributed.worker.memory.rebalance.measure": "managed_in_memory",
}):
    print("Active overrides:")
    print(f"  idle-timeout : {dask.config.get('distributed.scheduler.idle-timeout')}")
    print(f"  worker.daemon: {dask.config.get('distributed.worker.daemon')}")

    client, cluster, tmp = setup_dask_client(
        workload_type="mixed",
        max_workers=2,
        reserve_mem_gb=8.0,
        dashboard=False,
    )

    workers = client.scheduler_info()["workers"]
    print(f"Workers: {len(workers)}")

    data = da.random.random((1000, 1000), chunks=(200, 200))
    result = (data * 2).mean().compute()
    print(f"Result: {result:.6f}")

    client.close(); cluster.close()

## 4. Custom YAML Profiles

Save named configurations as YAML files for sharing across a team or project.
The `configs/cpu_profile.yaml` file ships with this recipe as an example.

In [ ]:
import yaml

# Load the example profile that ships with this recipe
profile_path = Path("../configs/cpu_profile.yaml")

if profile_path.exists():
    with open(profile_path) as f:
        profile_data = yaml.safe_load(f)

    print(f"Profile: {profile_data.get('name', 'unnamed')}")
    print(f"Description: {profile_data.get('description', '')}")
    for k, v in profile_data.items():
        if k not in ("name", "description", "tags", "notes"):
            print(f"  {k}: {v}")
else:
    print(f"Profile file not found at {profile_path} — creating a demo profile instead")
    profile_data = {
        "name": "demo_cpu",
        "workload_type": "cpu",
        "max_workers": 4,
        "reserve_mem_gb": 8.0,
        "spill_compression": "lz4",
    }

In [ ]:
# Create and save a custom I/O-heavy profile
io_profile = {
    "name": "io_heavy",
    "description": "High-throughput I/O with many threads",
    "workload_type": "io",
    "max_workers": 2,
    "reserve_mem_gb": 40.0,
    "spill_compression": "snappy",
    "tags": ["io-heavy", "multithreaded"],
    "notes": "Optimized for opening many NetCDF/Zarr files in parallel.",
}

out_path = Path("../configs/io_heavy_profile.yaml")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    yaml.dump(io_profile, f, default_flow_style=False, sort_keys=False)

print(f"Profile written to: {out_path}")
print(yaml.dump(io_profile, default_flow_style=False, sort_keys=False))

## 5. Profile Performance Comparison

Benchmark several profiles on the same workload to pick the best fit.

In [ ]:
profiles_to_test = ["development", "interactive"]
times = {}

for profile_name in profiles_to_test:
    try:
        client, cluster, _ = setup_dask_client(profile=profile_name, dashboard=False)

        t0 = time.perf_counter()
        x = da.random.random((2000, 2000), chunks=(500, 500))
        _ = (x + x.T).mean().compute()
        times[profile_name] = time.perf_counter() - t0

        client.close(); cluster.close()
        print(f"{profile_name:20s}  {times[profile_name]:.2f}s")
    except Exception as e:
        print(f"{profile_name:20s}  ERROR: {e}")
        times[profile_name] = float("inf")

if times:
    fastest = min(times, key=times.get)
    print(f"\nFastest: {fastest} ({times[fastest]:.2f}s)")

## Configuration Best Practices

**Priority order** (highest wins): explicit function params → environment variables → profile → built-in defaults.

**Profile selection guide:**

| Profile | Workload | When to use |
|---------|----------|-------------|
| `climate_analysis` | cpu | Large xarray reductions on HPC |
| `zarr_io_heavy` | io | Opening many Zarr/NetCDF files |
| `interactive` | mixed | Jupyter notebooks on a login node |
| `development` | mixed | Testing on a laptop or small VM |
| `production` | mixed | Long-running automated jobs |

**YAML profile tips:**
- Keep profiles in version control alongside your analysis scripts
- Use descriptive `description` and `notes` fields for future reference
- Validate with `dask-setup validate <name>` before deploying
- Use `based_on: parent_profile` to inherit and override a base profile

**Memory tuning:**

```python
# Conservative — for shared systems or laptops
DaskSetupConfig(reserve_mem_gb=4.0, memory_target=0.60, memory_spill=0.70)

# Aggressive — large dedicated HPC node
DaskSetupConfig(reserve_mem_gb=8.0, memory_target=0.80, spill_compression="lz4")
```